# Análisis exploratorio - AquaLimpia S. A.

Este notebook presenta el análisis exploratorio del dataset de aguas residuales de **AquaLimpia S. A.**.

El objetivo es evaluar el desempeño de las plantas de tratamiento, calcular indicadores de eficiencia, revisar el cumplimiento normativo, identificar alertas operativas y generar información útil para las áreas de Operaciones y Gestión Ambiental.

> Este notebook utiliza los archivos iniciales del proyecto:
>
> - `main.py`
> - `src/funciones_aqualimpia.py`
> - `dashboard/dashboard.py`
> - `data/raw/dataset_set_A_aguas_residuales.xlsx`
> - `outputs/`


## 1. Objetivos del análisis

### Objetivo general

Analizar el desempeño de las plantas de tratamiento de AquaLimpia S. A. mediante Python, indicadores, visualizaciones y reportes, con el fin de apoyar decisiones operativas y ambientales.

### Objetivos específicos

- Cargar y revisar el dataset oficial de aguas residuales.
- Calcular la eficiencia de remoción de DBO.
- Evaluar el cumplimiento normativo por planta.
- Identificar alertas operativas.
- Revisar la calidad de los datos.
- Generar reportes para Operaciones y Gestión Ambiental.
- Presentar resultados mediante tablas y visualizaciones.


## 2. Configuración inicial

En esta sección se importan las librerías necesarias y se configura la ruta del proyecto.  
El notebook está pensado para guardarse dentro de la carpeta `notebooks/`.

La estructura esperada del proyecto es:

```text
aqualimpia-ciencia-datos/
│
├── data/
│   └── raw/
│       └── dataset_set_A_aguas_residuales.xlsx
│
├── src/
│   ├── __init__.py
│   └── funciones_aqualimpia.py
│
├── dashboard/
│   └── dashboard.py
│
├── outputs/
├── notebooks/
│   └── analisis_aqualimpia.ipynb
│
├── main.py
├── README.md
└── requirements.txt
```


In [ ]:
# ============================================================
# Configuración inicial del notebook
# ============================================================

from pathlib import Path
import sys
import os
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from joblib import dump

# Detectar la raíz del proyecto.
# Si el notebook se ejecuta desde la carpeta notebooks, subimos un nivel.
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Agregar la raíz del proyecto al path para poder importar desde src/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Raíz del proyecto:", PROJECT_ROOT)


## 3. Carga de funciones reutilizables

El proyecto utiliza código modular.  
Las funciones principales se encuentran en el archivo externo:

```text
src/funciones_aqualimpia.py
```

Esto permite reutilizar el análisis, evitar duplicación de código y mantener un flujo reproducible.


In [ ]:
# Importación de funciones creadas en el proyecto

from src.funciones_aqualimpia import (
    cargar_datos,
    preparar_datos,
    resumen_por_planta,
    intervalo_confianza_dbo_salida,
    evaluar_calidad_datos,
    exportar_reportes,
    guardar_resultados
)

print("Funciones importadas correctamente.")


## 4. Carga del dataset

Se carga el archivo oficial del caso:

```text
data/raw/dataset_set_A_aguas_residuales.xlsx
```


In [ ]:
# Ruta del archivo de datos

RUTA_DATOS = PROJECT_ROOT / "data" / "raw" / "dataset_set_A_aguas_residuales.xlsx"

if not RUTA_DATOS.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo: {RUTA_DATOS}\n"
        "Verifica que el dataset esté dentro de data/raw/"
    )

df_original = cargar_datos(RUTA_DATOS)

print("Dataset cargado correctamente.")
print("Filas:", df_original.shape[0])
print("Columnas:", df_original.shape[1])

df_original.head()


## 5. Exploración inicial del dataset

En esta etapa se revisa la estructura general de los datos: columnas, tipos de datos, valores nulos y registros duplicados.


In [ ]:
# Columnas del dataset

df_original.columns


In [ ]:
# Información general del dataset

df_original.info()


In [ ]:
# Estadística descriptiva de variables numéricas

df_original.describe()


In [ ]:
# Revisión de valores nulos y duplicados

nulos = df_original.isnull().sum()
duplicados = df_original.duplicated().sum()

print("Valores nulos por columna:")
print(nulos)

print("\nRegistros duplicados:", duplicados)


## 6. Preparación de datos

Se prepara el dataset mediante la función `preparar_datos()`.

Esta función realiza tres acciones principales:

1. Convierte `fecha_registro` a formato fecha.
2. Calcula la eficiencia de remoción de DBO.
3. Crea una alerta operativa cuando existe incumplimiento normativo o eficiencia menor al 85 %.


In [ ]:
# Preparación del dataset

df = preparar_datos(df_original)

print("Datos preparados correctamente.")
df.head()


In [ ]:
# Verificar nuevas columnas creadas

df.columns


## 7. Cálculo de indicadores generales

Se calculan indicadores globales para describir el desempeño general de AquaLimpia S. A.


In [ ]:
# Indicadores generales del proyecto

registros = len(df)
columnas_originales = len(df_original.columns)
fecha_inicio = df["fecha_registro"].min()
fecha_fin = df["fecha_registro"].max()

cumplimiento_global = df["cumplimiento_norma"].mean() * 100
incumplimiento_global = 100 - cumplimiento_global

dbo_entrada_promedio = df["DBO_entrada_mg_L"].mean()
dbo_salida_promedio = df["DBO_salida_mg_L"].mean()
eficiencia_promedio = df["eficiencia_DBO_pct"].mean()

correlacion_dbo = df["DBO_entrada_mg_L"].corr(df["DBO_salida_mg_L"])

indicadores_generales = pd.DataFrame({
    "Indicador": [
        "Registros analizados",
        "Columnas originales",
        "Fecha inicial",
        "Fecha final",
        "Cumplimiento global (%)",
        "Incumplimiento global (%)",
        "DBO entrada promedio (mg/L)",
        "DBO salida promedio (mg/L)",
        "Eficiencia promedio remoción DBO (%)",
        "Correlación DBO entrada / DBO salida"
    ],
    "Resultado": [
        registros,
        columnas_originales,
        fecha_inicio.strftime("%d-%m-%Y"),
        fecha_fin.strftime("%d-%m-%Y"),
        round(cumplimiento_global, 2),
        round(incumplimiento_global, 2),
        round(dbo_entrada_promedio, 2),
        round(dbo_salida_promedio, 2),
        round(eficiencia_promedio, 2),
        round(correlacion_dbo, 3)
    ]
})

indicadores_generales


## 8. Resumen por planta

Se agrupan los datos por planta de tratamiento para comparar desempeño operacional y ambiental.


In [ ]:
# Resumen por planta usando la función modular

resumen = resumen_por_planta(df)

# Ajustar cumplimiento a porcentaje si está entre 0 y 1
resumen_mostrar = resumen.copy()

if resumen_mostrar["cumplimiento_pct"].max() <= 1:
    resumen_mostrar["cumplimiento_pct"] = resumen_mostrar["cumplimiento_pct"] * 100

# Agregar número de alertas por planta
alertas_por_planta = (
    df.groupby("planta")["alerta_operativa"]
    .apply(lambda x: (x == "Alerta").sum())
    .reset_index(name="alertas")
)

resumen_mostrar = resumen_mostrar.merge(alertas_por_planta, on="planta", how="left")

# Redondear valores
columnas_numericas = resumen_mostrar.select_dtypes(include="number").columns
resumen_mostrar[columnas_numericas] = resumen_mostrar[columnas_numericas].round(2)

resumen_mostrar


## 9. Interpretación del resumen por planta

A partir del resumen por planta se identifica qué unidad presenta menor cumplimiento normativo y cuál registra más alertas operativas.


In [ ]:
# Identificación de plantas críticas

planta_menor_cumplimiento = resumen_mostrar.sort_values("cumplimiento_pct").iloc[0]
planta_mas_alertas = resumen_mostrar.sort_values("alertas", ascending=False).iloc[0]

print("Planta con menor cumplimiento normativo:")
print(planta_menor_cumplimiento[["planta", "cumplimiento_pct"]])

print("\nPlanta con más alertas operativas:")
print(planta_mas_alertas[["planta", "alertas"]])


## 10. Intervalo de confianza para DBO de salida

Se calcula un intervalo de confianza del 95 % para la media de la DBO de salida.  
Este cálculo utiliza **SciPy**, lo que permite incorporar análisis estadístico al proyecto.


In [ ]:
# Intervalo de confianza para la media de DBO de salida

ic_dbo = intervalo_confianza_dbo_salida(df)

# Mostrar resultado redondeado
ic_dbo_redondeado = {
    "media_DBO_salida": round(ic_dbo["media_DBO_salida"], 2),
    "IC95_inferior": round(ic_dbo["IC95_inferior"], 2),
    "IC95_superior": round(ic_dbo["IC95_superior"], 2)
}

ic_dbo_redondeado


## 11. Evaluación de calidad de datos

La calidad de datos es fundamental porque los resultados pueden influir en reportes ambientales, decisiones operativas y priorización de inversiones.

En esta etapa se revisan:

- Filas y columnas.
- Valores nulos.
- Registros duplicados.
- Posibles inconsistencias entre DBO de salida y cumplimiento normativo.


In [ ]:
# Evaluación de calidad de datos usando la función modular

calidad = evaluar_calidad_datos(df)

calidad


In [ ]:
# Presentación de calidad de datos en formato tabular

calidad_resumen = pd.DataFrame({
    "Aspecto evaluado": [
        "Filas",
        "Columnas",
        "Registros duplicados",
        "Registros con DBO salida <= 30 e incumplimiento"
    ],
    "Resultado": [
        calidad.get("filas"),
        calidad.get("columnas"),
        calidad.get("duplicados"),
        calidad.get("registros_DBO_menor_igual_30_incumplen")
    ]
})

calidad_resumen


## 12. Visualización 1: Cumplimiento normativo por planta

Este gráfico permite comparar el porcentaje de cumplimiento normativo entre las plantas de tratamiento.


In [ ]:
# Gráfico de cumplimiento normativo por planta

plt.figure(figsize=(8, 5))
plt.bar(resumen_mostrar["planta"], resumen_mostrar["cumplimiento_pct"])
plt.title("Cumplimiento normativo por planta")
plt.xlabel("Planta")
plt.ylabel("Cumplimiento (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 13. Visualización 2: DBO de salida promedio por planta

Este gráfico permite observar diferencias en la calidad del efluente tratado entre plantas.


In [ ]:
# Gráfico de DBO de salida promedio por planta

plt.figure(figsize=(8, 5))
plt.bar(resumen_mostrar["planta"], resumen_mostrar["DBO_salida_promedio"])
plt.title("DBO de salida promedio por planta")
plt.xlabel("Planta")
plt.ylabel("DBO salida promedio (mg/L)")
plt.tight_layout()
plt.show()


## 14. Visualización 3: Relación entre DBO de entrada y DBO de salida

Este gráfico permite observar la relación entre la carga contaminante de entrada y la DBO del efluente tratado.


In [ ]:
# Gráfico de dispersión: DBO entrada vs DBO salida

plt.figure(figsize=(8, 5))
plt.scatter(df["DBO_entrada_mg_L"], df["DBO_salida_mg_L"])
plt.title("Relación entre DBO de entrada y DBO de salida")
plt.xlabel("DBO entrada (mg/L)")
plt.ylabel("DBO salida (mg/L)")
plt.tight_layout()
plt.show()

print("Correlación:", round(correlacion_dbo, 3))


## 15. Visualización 4: Evolución temporal de la DBO de salida

Este gráfico permite revisar el comportamiento temporal de la DBO de salida por planta.


In [ ]:
# Evolución temporal de DBO de salida por planta

plt.figure(figsize=(10, 5))

for planta in df["planta"].unique():
    datos_planta = df[df["planta"] == planta].sort_values("fecha_registro")
    plt.plot(
        datos_planta["fecha_registro"],
        datos_planta["DBO_salida_mg_L"],
        marker="o",
        linewidth=1,
        label=planta
    )

plt.title("Evolución temporal de DBO de salida")
plt.xlabel("Fecha")
plt.ylabel("DBO salida (mg/L)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 16. Registros con alerta operativa

Se identifican los registros clasificados como alerta.  
Un registro se considera alerta cuando:

- No cumple la norma, o
- La eficiencia de remoción de DBO es menor al 85 %.


In [ ]:
# Registros con alerta operativa

alertas = df[df["alerta_operativa"] == "Alerta"]

print("Cantidad de alertas:", len(alertas))

alertas[[
    "fecha_registro",
    "planta",
    "caudal_entrada_m3_d",
    "DBO_entrada_mg_L",
    "DBO_salida_mg_L",
    "eficiencia_DBO_pct",
    "cumplimiento_norma",
    "alerta_operativa"
]].head(20)


## 17. Exportación de reportes

Se generan reportes diferenciados para dos áreas:

### Área de Operaciones

Incluye información operativa como fecha, planta, caudal, DBO de entrada, DBO de salida, energía de aireación, lodos generados, eficiencia y alerta operativa.

### Área de Gestión Ambiental

Incluye fecha, planta, DBO de salida, cumplimiento normativo y alerta operativa.


In [ ]:
# Exportación de reportes usando la función modular

CARPETA_SALIDA = PROJECT_ROOT / "outputs"
CARPETA_SALIDA.mkdir(exist_ok=True)

exportar_reportes(df, resumen, CARPETA_SALIDA)

print("Reportes exportados en:", CARPETA_SALIDA)


## 18. Guardado de resultados

Se guardan los resultados principales en:

- `calidad_datos.json`
- `resultados_aqualimpia.joblib`

El archivo Joblib permite reutilizar resultados posteriormente sin recalcular todo el análisis.


In [ ]:
# Guardado de resultados en JSON y Joblib

resultados = {
    "indicadores_generales": indicadores_generales.to_dict(orient="records"),
    "resumen_por_planta": resumen_mostrar.to_dict(orient="records"),
    "calidad_datos": calidad,
    "intervalo_confianza_DBO_salida": ic_dbo_redondeado
}

# JSON
with open(CARPETA_SALIDA / "calidad_datos.json", "w", encoding="utf-8") as archivo:
    json.dump(calidad, archivo, indent=4, ensure_ascii=False)

# Joblib
guardar_resultados(resultados, CARPETA_SALIDA / "resultados_aqualimpia.joblib")

print("Resultados guardados correctamente.")


## 19. Resultados principales del análisis

A partir del análisis realizado se obtienen los siguientes resultados generales.


In [ ]:
# Mostrar indicadores generales

indicadores_generales


In [ ]:
# Mostrar resumen por planta

resumen_mostrar


## 20. Interpretación de resultados

El análisis permite observar diferencias de desempeño entre las plantas de tratamiento.  
El cumplimiento global debe analizarse con precaución, ya que puede existir una posible inconsistencia entre la DBO de salida y la variable de cumplimiento normativo.

La relación entre DBO de entrada y DBO de salida permite evaluar si la carga contaminante inicial influye en la calidad del efluente tratado. Una correlación positiva indica que, cuando aumenta la DBO de entrada, también tiende a aumentar la DBO de salida.

La identificación de alertas operativas permite priorizar revisiones técnicas en plantas con mayor riesgo de incumplimiento.


In [ ]:
# Conclusión automática con base en resultados calculados

print("Conclusión automática del análisis")
print("----------------------------------")

print(
    f"La planta con menor cumplimiento normativo es "
    f"{planta_menor_cumplimiento['planta']}, con "
    f"{planta_menor_cumplimiento['cumplimiento_pct']:.2f}%."
)

print(
    f"La planta con más alertas operativas es "
    f"{planta_mas_alertas['planta']}, con "
    f"{int(planta_mas_alertas['alertas'])} alertas."
)

print(
    f"La DBO de salida promedio es {dbo_salida_promedio:.2f} mg/L."
)

print(
    f"La eficiencia promedio de remoción de DBO es {eficiencia_promedio:.2f}%."
)

print(
    f"La correlación entre DBO de entrada y DBO de salida es {correlacion_dbo:.3f}."
)


## 21. Limitaciones del análisis

El análisis presenta las siguientes limitaciones:

- No incluye variables climáticas, como lluvia o temperatura.
- No incorpora información sobre mantenciones o fallas de equipos.
- No registra dosificación química.
- No se conoce completamente la regla utilizada para definir `cumplimiento_norma`.
- El análisis permite identificar patrones, pero no demostrar causalidad.
- Los registros con DBO de salida menor o igual a 30 mg/L y cumplimiento igual a 0 deben ser revisados antes de usar el análisis como evidencia formal.


## 22. Conclusión final

El proyecto permite analizar el desempeño de las plantas de tratamiento de AquaLimpia S. A. mediante un flujo reproducible basado en Python.

El uso de scripts reutilizables, reportes automatizados, análisis de calidad de datos y visualizaciones permite transformar los registros operacionales en información útil para la toma de decisiones.

La solución contribuye a:

- Detectar plantas con mayor riesgo operativo.
- Apoyar reportes ambientales.
- Priorizar acciones correctivas.
- Mejorar la trazabilidad del análisis.
- Facilitar la actualización del proyecto con nuevos datos.


## 23. Ejecución complementaria del dashboard

Además del notebook, el proyecto incluye un dashboard en Streamlit.

Para ejecutarlo desde la carpeta principal del proyecto:

```bash
py -m streamlit run dashboard\dashboard.py
```

Para ejecutar el análisis principal:

```bash
py main.py
```
